<h2> 
    
Text Analysis in Julia, with [TextAnalysis.jl]( https://juliatext.github.io/TextAnalysis.jl/dev/ )  

</h2><h3>Aliza Rogers</h3>

In [20]:
using TextAnalysis

<p>To begin, we will need documents of text to process and analyze. I'll only use works of fiction.</p>       
      
<p>  

All ebook files are from [Project Gutenberg]( https://www.gutenberg.org/ ), a libary of free e-books that are in the public domain. </p>

The ebooks may be accessed individually, as documents. They are also all accessable via the corpus.

In [21]:
# get_document loads the file at the path name given.
# If a Project Gutenberg book is given, the content before and after the markers will be stripped.

function get_document(path::String)
    #reading in the file from the given path
    text = open(path) do file
        read(file, String)
    end
    # Project Gutenberg files typically start and end with these markers
    start_regex = r"\*\*\* START OF THIS PROJECT GUTENBERG EBOOK.*?\*\*\*"
    end_regex = r"\*\*\* END OF THIS PROJECT GUTENBERG EBOOK.*?\*\*\*"

    start_match = findfirst(start_regex, text)
    end_match = findfirst(end_regex, text)

    # if the ebook contains Gutenberg markers, extract the content between them
    if !(isnothing(start_match) || isnothing(end_match))
        text = text[start_match.stop:end_match.start]
    end

    return StringDocument(text)
end

get_document (generic function with 1 method)

In [22]:
# getting the path to the directory containing the Project Gutenberg books
book_path = pwd() * "/ebooks"

##### getting individual ebooks prepared #####

#Alice's Adventures in Wonderland - Lewis Carroll
wonderland = get_document(book_path * "/11-0.txt")

title!(wonderland, "Alice's Adventures in Wonderland")
author!(wonderland, "Lewis Carroll")
timestamp!(wonderland, "1865")


# The Wonderful Wizard of Oz - L. Frank Baum
oz = get_document(book_path * "/55-0.txt")

title!(oz, "The Wonderful Wizard of Oz")
author!(oz, "L. Frank Baum")
timestamp!(oz, "1900")


# The Secret Garden - Frances Hodgson Burnett
secret_garden = get_document(book_path * "/113-0.txt")

title!(secret_garden, "The Secret Garden")
author!(secret_garden, "Frances Hodgson Burnett")
timestamp!(secret_garden, "1911")


# Treasure Island - Robert Louis Stevenson
treasure_island = get_document(book_path * "/120-0.txt")

title!(treasure_island, "Treasure Island")
author!(treasure_island, "Robert Louis Stevenson")
timestamp!(treasure_island, "1883")


# The Jungle Book - Rudyard Kipling
jungle_book = get_document(book_path * "/236-0.txt")

title!(jungle_book, "The Jungle Book")
author!(jungle_book, "Rudyard Kipling")
timestamp!(jungle_book, "1894")


# The Adventures of Pinocchio - C. Collodi
pinocchio = get_document(book_path * "/500-0.txt")

title!(pinocchio, "The Adventures of Pinocchio")
author!(pinocchio, "C. Collodi")
timestamp!(pinocchio, "1883")


# Peter and Wendy (Peter Pan) - J. M. Barrie
peter_pan = get_document(book_path * "/26654-8.txt")

title!(peter_pan, "Peter and Wendy")
author!(peter_pan, "J. M. Barrie")
timestamp!(peter_pan, "1911")

doc_list = [wonderland, oz, secret_garden, treasure_island, jungle_book, pinocchio, peter_pan]

7-element Vector{StringDocument{String}}:
 A StringDocument{String}
 A StringDocument{String}
 A StringDocument{String}
 A StringDocument{String}
 A StringDocument{String}
 A StringDocument{String}
 A StringDocument{String}

<p> Next, we will create a corpus. A corpus is a collection of documents, to be analyzed. The most common words within the corpus will be displayed between each step of cleaning it up. This will be very useful for understanding <i>why</i> clean up is important.</p>

In [23]:
corpus = Corpus(doc_list)

using OrderedCollections

update_lexicon!(corpus)

lexicon(corpus)

ordered_lexicon = OrderedDict(sort(collect(lexicon(corpus)), by = x -> x[2], rev = true))

OrderedDict{String, Int64} with 20809 entries:
  ","    => 25558
  "the"  => 18323
  "and"  => 13005
  "to"   => 8897
  "“"    => 8090
  "a"    => 7809
  "’"    => 7180
  "of"   => 6982
  "”"    => 6720
  "I"    => 6414
  "he"   => 5185
  "was"  => 5137
  "in"   => 4454
  "that" => 3763
  "it"   => 3494
  "you"  => 3445
  "his"  => 3306
  "'"    => 3234
  "had"  => 3074
  "as"   => 2925
  "she"  => 2850
  "!"    => 2694
  "said" => 2665
  "with" => 2586
  "s"    => 2409
  ⋮      => ⋮

<b> Notice that quite a few of the most popular words are punctuation marks. To fix this, we can clean up the corpus by removing punctuation.</b>

In [24]:
prepare!(corpus, strip_punctuation | strip_corrupt_utf8)

remove_case!(corpus) # making all letters lowercase

update_lexicon!(corpus)
ordered_lexicon = OrderedDict(sort(collect(lexicon(corpus)), by = x -> x[2], rev = true))


OrderedDict{String, Int64} with 15304 entries:
  "the"  => 19873
  "and"  => 13699
  "to"   => 9016
  "a"    => 8001
  "of"   => 7133
  "he"   => 6232
  "i"    => 5592
  "was"  => 5196
  "in"   => 4748
  "it"   => 4577
  "that" => 3966
  "you"  => 3892
  "she"  => 3502
  "his"  => 3407
  "as"   => 3118
  "had"  => 3096
  "said" => 2956
  "with" => 2673
  "for"  => 2541
  "but"  => 2438
  "on"   => 2366
  "they" => 2348
  "at"   => 2325
  "her"  => 2292
  "not"  => 2164
  ⋮      => ⋮

<b> These words are very general, and they don't tell much about the text. In NLP, these are called "stop words". Stop words usually include articles, prepositions, pronouns, and other similar words. </b>

In [25]:

prepare!(corpus, strip_stopwords)

update_lexicon!(corpus)
ordered_lexicon = OrderedDict(sort(collect(lexicon(corpus)), by = x -> x[2], rev = true))

OrderedDict{String, Int64} with 14899 entries:
  "little"    => 1098
  "mary"      => 673
  "time"      => 653
  "looked"    => 516
  "head"      => 512
  "dont"      => 448
  "cried"     => 447
  "pinocchio" => 430
  "eyes"      => 426
  "look"      => 420
  "alice"     => 386
  "peter"     => 380
  "th"        => 372
  "ill"       => 357
  "heard"     => 354
  "answered"  => 350
  "found"     => 348
  "dorothy"   => 347
  "tell"      => 341
  "day"       => 335
  "wendy"     => 334
  "round"     => 334
  "door"      => 311
  "mother"    => 306
  "hand"      => 306
  ⋮           => ⋮